# 02 - Multi-Table Relational EDA & Aggregations

**Track 2 Lead**: Complete
**Course**: UC Berkeley DATASCI 207 (Machine Learning at Scale)  
**Target Deliverable**: Project Milestone & Final Submission  
**Scope**: Multi-table relational joins across `bureau.csv`, `previous_application.csv`, `POS_CASH_balance.csv`, `installments_payments.csv`, and `credit_card_balance.csv`.

---

### Environment & Data Ingestion (Google Colab)
Run the cell below if executing in Google Colab to mount Google Drive and pull dataset files to `/content/local_data`.


In [1]:
# NOTE:
# If you get 'Directory not found', add the shared folder to your My Drive:
# 1. Go to Google Drive (https://drive.google.com).
# 2. Click 'Shared with me' on the left menu.
# 3. Right-click the 'DATASCI207_finalproject' folder.
# 4. Select Organize -> Add shortcut (or press Shift + Z).
# 5. Choose 'My Drive' and click Add.

try:
    from google.colab import drive
    import shutil, os

    # Mount Google Drive
    drive.mount('/content/drive')

    # Copy shared project folder to local Colab storage
    src = "/content/drive/MyDrive/DATASCI207_finalproject"
    dst = "/content/DATASCI207_finalproject"

    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.gdoc', '*.gsheet', '*.gslides'))
    print("Downloaded files:", os.listdir(dst))
except ImportError:
    print("Running in local environment; skipping Google Drive mount.")


Running in local environment; skipping Google Drive mount.


### Cleaned Static Baseline Ingestion (Task 1)

Loads the cleaned and engineered static applicant dataset (`application_train_cleaned.parquet`) produced by Task 1.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve repo root regardless of where the kernel's cwd ends up

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load Task 1 cleaned application matrix
data_path = Path('/content/DATASCI207_finalproject/data/processed/application_train_cleaned.parquet')
if not data_path.exists():
    data_path = PROJECT_ROOT / 'data' / 'processed' / 'application_train_cleaned.parquet'

df_cleaned = pd.read_parquet(data_path)
print(f'Loaded cleaned data: {df_cleaned.shape[0]:,} rows x {df_cleaned.shape[1]:,} columns')

# Raw relational tables directory (bureau, previous_application, etc.)
raw_dir = Path('/content/DATASCI207_finalproject/data/raw/home_credit_default_risk')
if not raw_dir.exists():
    raw_dir = PROJECT_ROOT / 'data' / 'raw' / 'home_credit_default_risk'
RAW_DIR = raw_dir
if RAW_DIR.exists():
    print('Available raw tables:', [f.name for f in RAW_DIR.glob('*.csv')])

Loaded cleaned data: 307,511 rows x 131 columns
Available raw tables: ['application_test.csv', 'HomeCredit_columns_description.csv', 'POS_CASH_balance.csv', 'credit_card_balance.csv', 'installments_payments.csv', 'application_train.csv', 'bureau.csv', 'previous_application.csv', 'bureau_balance.csv', 'sample_submission.csv']


In [3]:
from sklearn.model_selection import train_test_split

# 70 / 10 / 20 stratified split
_train_plus_val_idx, test_idx = train_test_split(
    df_cleaned.index, test_size=0.20, stratify=df_cleaned['TARGET'], random_state=1234
)
train_idx, val_idx = train_test_split(
    _train_plus_val_idx, test_size=0.125, # 0.125 of the remaining 80% = 10% of the whole set
    stratify=df_cleaned.loc[_train_plus_val_idx, 'TARGET'], random_state=1234
)

split = pd.Series('train', index=df_cleaned.index)
split.loc[val_idx] = 'val'
split.loc[test_idx] = 'test'
df_cleaned['SPLIT'] = split.values

for name in ['train', 'val', 'test']:
    mask = df_cleaned['SPLIT'] == name
    print(f"{name:5s}: {mask.sum():>7,} rows default_rate={df_cleaned.loc[mask, 'TARGET'].mean():.4f}")

train: 215,257 rows default_rate=0.0807
val  :  30,751 rows default_rate=0.0807
test :  61,503 rows default_rate=0.0807


**Why the split happens here, before any further processing:** none of the aggregations below use `TARGET`,
so there's no leakage risk in the aggregated *values* themselves. But once we compute anything that
correlates an aggregated feature with `TARGET` (below), or select features based on statistics that touch
the label, those statistics must only ever see the **training** rows -- otherwise we'd be letting information
from what will become the validation/test set quietly influence decisions made upstream of modeling. Defining
`SPLIT` here, immediately after loading the cleaned matrix, means every `TARGET`-aware cell from this point
forward can filter to `SPLIT == 'train'` first. The default rate landing at an identical 8.07% across all
three partitions confirms the stratification worked as intended.

### Join Diagnostics: Evaluating Match Quality Across the Four Relational Tables

Before trusting the left join below, we check the three failure modes an instructor flagged for any multi-table join:

- **Duplicate matches** — would a table contribute more than one row per `SK_ID_CURR` after aggregation, silently inflating the merged row count?
- **Missing matches** — what share of applicants have *no* record in a given behavioral table (legitimate "no history", not an error)?
- **Bad matches** — do any behavioral records reference an `SK_ID_CURR` that doesn't exist in our application table at all (orphan keys), or carry a null/invalid key?

We answer these by reading just the `SK_ID_CURR` column of each raw file (cheap) and comparing it against the applicant universe, *before* any aggregation happens.

In [4]:
## Join diagnostics: read only SK_ID_CURR from each raw table and compare against the application universe
import gc
app_ids = set(df_cleaned['SK_ID_CURR'])
n_app = len(app_ids)

_tables_to_check = {
    'bureau.csv': 'bureau',
    'previous_application.csv': 'previous_application',
    'POS_CASH_balance.csv': 'POS_CASH_balance',
    'installments_payments.csv': 'installments_payments',
}

_diag_rows = []
for fname, label in _tables_to_check.items():
    ids = pd.read_csv(RAW_DIR / fname, usecols=['SK_ID_CURR'])['SK_ID_CURR']
    n_rows = len(ids)
    uniq_ids = ids.unique()
    uniq_set = set(uniq_ids)

    bad_keys = int(ids.isna().sum() + (ids.dropna() <= 0).sum())
    orphans = uniq_set - app_ids          # bad matches: table IDs with no application record
    covered = uniq_set & app_ids          # applicants this table actually covers

    _diag_rows.append({
        'table': label,
        'raw_rows': n_rows,
        'unique_ids': len(uniq_set),
        'rows_per_id': round(n_rows / len(uniq_set), 1),
        'orphan_ids (not in app table)': len(orphans),
        'orphan_%_of_table_ids': round(len(orphans) / len(uniq_set) * 100, 1),
        'applicant_coverage_%': round(len(covered) / n_app * 100, 1),
        'null_or_invalid_keys': bad_keys,
    })
    del ids, uniq_ids, uniq_set
    gc.collect()

join_diag_report = pd.DataFrame(_diag_rows)
join_diag_report

,table,raw_rows,unique_ids,rows_per_id,orphan_ids (not in app table),orphan_%_of_table_ids,applicant_coverage_%,null_or_invalid_keys
0,bureau,1716428,305811,5.6,42320,13.8,85.7,0
1,previous_application,1670214,338857,4.9,47800,14.1,94.6,0
2,POS_CASH_balance,10001358,337252,29.7,47808,14.2,94.1,0
3,installments_payments,13605401,339587,40.1,47944,14.1,94.8,0


**Reading the diagnostics:**
- **No duplicate-match risk**: every table is aggregated to one row per `SK_ID_CURR` *before* merging (Section below), so `rows_per_id` > 1 here just reflects normal one-to-many loan/installment history, not a join bug — we re-verify uniqueness right before the merge.
- **Missing matches are real and expected**: 5–15% of applicants have no record in any single table (e.g. first-time borrowers have no `bureau.csv` history). We fill count-style aggregates with 0 ("no history") but deliberately leave rate/mean aggregates as `NaN` (Section 2, point 5) rather than 0, since a 0% delay rate is not the same claim as "no data" — see the missingness note after the merge below.
- **Bad matches ("orphans") are ~14% in every table and are not a data quality bug**: they line up almost exactly with the 48,744 rows of `application_test.csv` (Kaggle's held-out test applicants), which share the same relational tables as `application_train.csv` but naturally aren't in our training application table. The left join below correctly excludes them.

In [5]:
import gc

def downcast(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category')
    return df

def mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1e6



In [6]:
## aggregation 1 - bureau.csv - external credit bureau lines
bureau = pd.read_csv(
    RAW_DIR / 'bureau.csv',
    usecols=['SK_ID_CURR', 'CREDIT_ACTIVE', 'AMT_CREDIT_SUM_OVERDUE',
             'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM_DEBT', 'DAYS_CREDIT']
)
print(f"bereau.csv loaded: {bureau.shape[0]:,} rows, {mb(bureau):.1f} MB")
bureau = downcast(bureau)
print(f"after downcast: {mb(bureau):.1f} MB")

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_ACTIVE_LINES=('CREDIT_ACTIVE', lambda s: (s == 'Active').sum()),
    BUREAU_TOTAL_LINES=('CREDIT_ACTIVE', 'count'),
    BUREAU_TOTAL_OVERDUE_DEBT=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUREAU_MAX_DAYS_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_OLDEST_CREDIT_DAYS=('DAYS_CREDIT', 'min'),
).reset_index()

del bureau
gc.collect()
bureau_agg.head()

bereau.csv loaded: 1,716,428 rows, 92.7 MB
after downcast: 42.9 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_87644/1900776898.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,BUREAU_ACTIVE_LINES,BUREAU_TOTAL_LINES,BUREAU_TOTAL_OVERDUE_DEBT,BUREAU_MAX_DAYS_OVERDUE,BUREAU_TOTAL_DEBT,BUREAU_OLDEST_CREDIT_DAYS
0,100001,3,7,0.0,0,596686.5,-1572
1,100002,2,8,0.0,0,245781.0,-1437
2,100003,1,4,0.0,0,0.0,-2586
3,100004,0,2,0.0,0,0.0,-1326
4,100005,2,3,0.0,0,568408.5,-373


In [7]:
## aggregation 2 - previous_application.csv - prior home credit applications
prev = pd.read_csv(
    RAW_DIR / 'previous_application.csv',
    usecols=['SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'AMT_DOWN_PAYMENT', 'AMT_CREDIT']
)
print(f"previous_application.csv loaded: {prev.shape[0]:,} rows, {mb(prev):.1f} MB")
prev = downcast(prev)

prev['DOWN_PAYMENT_PCT'] = prev['AMT_DOWN_PAYMENT'] / prev['AMT_CREDIT']
prev['DOWN_PAYMENT_PCT'] = prev['DOWN_PAYMENT_PCT'].replace([np.inf, -np.inf], np.nan)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_REFUSAL_COUNT=('NAME_CONTRACT_STATUS', lambda s: (s == 'Refused').sum()),
    PREV_APPLICATION_COUNT=('NAME_CONTRACT_STATUS', 'count'),
    PREV_APPROVAL_RATE=('NAME_CONTRACT_STATUS', lambda s: (s == 'Approved').mean()),
    PREV_MEAN_DOWN_PAYMENT_PCT=('DOWN_PAYMENT_PCT', 'mean'),
).reset_index()

del prev
gc.collect()
prev_agg.head()

previous_application.csv loaded: 1,670,214 rows, 66.6 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_87644/1900776898.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,PREV_REFUSAL_COUNT,PREV_APPLICATION_COUNT,PREV_APPROVAL_RATE,PREV_MEAN_DOWN_PAYMENT_PCT
0,100001,0,1,1.0,0.105940
1,100002,0,1,1.0,0.000000
2,100003,0,3,1.0,0.050585
3,100004,0,1,1.0,0.241719
4,100005,0,2,0.5,0.111173


In [8]:
## aggregation 3 - POS_CASH.csv - monthly POS_CASH balances
pos = pd.read_csv(
    RAW_DIR / 'POS_CASH_balance.csv',
    usecols=['SK_ID_CURR', 'MONTHS_BALANCE', 'SK_DPD', 'SK_DPD_DEF', 'NAME_CONTRACT_STATUS',]
)
print(f"POS_CASH_balance.csv loaded: {pos.shape[0]:,} rows, {mb(pos):.1f} MB")
pos = downcast(pos)

pos_agg = pos.groupby('SK_ID_CURR').agg(
    POS_MONTH_COUNT=('MONTHS_BALANCE', 'count'), # trajectory length
    POS_MOST_RECENT_MONTH=('MONTHS_BALANCE', 'max'), # closest to 0 = most recent (bugfix: was 'count', a duplicate of POS_MONTH_COUNT)
    POS_MEAN_DPD=('SK_DPD', 'mean'),
    POS_MAX_DPD=('SK_DPD', 'max'),
    POS_MEAN_DPD_DEF=('SK_DPD_DEF', 'mean'),
    POS_COMPLETED_COUNT=('NAME_CONTRACT_STATUS', lambda s: (s == 'Completed').sum()),
    POS_ACTIVE_COUNT=('NAME_CONTRACT_STATUS', lambda s: (s == 'Active').sum()),
).reset_index()

del pos
gc.collect()
pos_agg.head()

POS_CASH_balance.csv loaded: 10,001,358 rows, 462.4 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_87644/1900776898.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,POS_MONTH_COUNT,POS_MOST_RECENT_MONTH,POS_MEAN_DPD,POS_MAX_DPD,POS_MEAN_DPD_DEF,POS_COMPLETED_COUNT,POS_ACTIVE_COUNT
0,100001,9,-53,0.777778,7,0.777778,2,7
1,100002,19,-1,0.000000,0,0.000000,0,19
2,100003,28,-18,0.000000,0,0.000000,2,26
3,100004,4,-24,0.000000,0,0.000000,1,3
4,100005,11,-15,0.000000,0,0.000000,1,9


In [9]:
## aggregation 4 - installments_payments.csv - big data set
inst = pd.read_csv(
    RAW_DIR / 'installments_payments.csv',
    usecols=['SK_ID_CURR', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']
)
print(f"installments_payments.csv loaded: {inst.shape[0]:,} rows, {mb(inst):.1f} MB")
inst = downcast(inst)

# positive DELAY_DAYS = paid late - negative/zero = on time or early
inst['DELAY_DAYS'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['UNDERPAID'] = inst['AMT_PAYMENT'] < inst['AMT_INSTALMENT']

inst_agg = inst.groupby('SK_ID_CURR').agg(
    INST_DELAY_MEAN=('DELAY_DAYS', 'mean'),
    INST_DELAY_MAX=('DELAY_DAYS', 'max'),
    INST_UNDERPAYMENT_FREQ=('UNDERPAID', 'mean'), # fraction of installments underpaid
    INST_TOTAL_PAYMENTS=('AMT_PAYMENT', 'count'),
).reset_index()

del inst
gc.collect()
inst_agg.head()

installments_payments.csv loaded: 13,605,401 rows, 544.2 MB


,SK_ID_CURR,INST_DELAY_MEAN,INST_DELAY_MAX,INST_UNDERPAYMENT_FREQ,INST_TOTAL_PAYMENTS
0,100001,-7.285714,11.0,0.0,7
1,100002,-20.421053,-12.0,0.0,19
2,100003,-7.160000,-1.0,0.0,25
3,100004,-7.666667,-3.0,0.0,3
4,100005,-23.555555,1.0,0.0,9


### Memory & Scale: Downcasting vs. Row Sampling

`installments_payments.csv` is our largest and most memory-constrained table (13.6M rows). We chose
column pruning + dtype downcasting + one-table-at-a-time processing with explicit `gc.collect()` to fit it
in Colab's 12.7 GB RAM. This cell empirically tests sampling against our current approach; Spark/Dask is
discussed afterward since neither is in `requirements.txt` for this project.

In [10]:
## Self-contained comparison: does randomly sampling rows save more memory than downcasting,
## and at what cost to the aggregated features
_cols = ['SK_ID_CURR', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']
_inst_full = pd.read_csv(RAW_DIR / 'installments_payments.csv', usecols=_cols)
_native_mb = mb(_inst_full)
_inst_full = downcast(_inst_full)
_downcast_mb = mb(_inst_full)
print(f"Full table:   native {_native_mb:,.1f} MB -> downcast {_downcast_mb:,.1f} MB "
      f"({(1 - _downcast_mb / _native_mb) * 100:.1f}% smaller), 0% accuracy loss")

_inst_full['DELAY_DAYS'] = _inst_full['DAYS_ENTRY_PAYMENT'] - _inst_full['DAYS_INSTALMENT']
_inst_full['UNDERPAID'] = _inst_full['AMT_PAYMENT'] < _inst_full['AMT_INSTALMENT']
_agg_full = _inst_full.groupby('SK_ID_CURR').agg(
    INST_DELAY_MEAN=('DELAY_DAYS', 'mean'),
    INST_DELAY_MAX=('DELAY_DAYS', 'max'),
    INST_UNDERPAYMENT_FREQ=('UNDERPAID', 'mean'),
    INST_TOTAL_PAYMENTS=('AMT_PAYMENT', 'count'),
)

## Alternative: keep only a random 20% sample of ROWS (the cheapest possible sampling strategy)
_rng = np.random.default_rng(42)
_frac = 0.20
_sample_mask = _rng.random(len(_inst_full)) < _frac
_inst_sample = _inst_full.loc[_sample_mask, ['SK_ID_CURR', 'DELAY_DAYS', 'UNDERPAID', 'AMT_PAYMENT']]
_sample_mb = mb(_inst_sample)
_agg_sample = _inst_sample.groupby('SK_ID_CURR').agg(
    INST_DELAY_MEAN=('DELAY_DAYS', 'mean'),
    INST_DELAY_MAX=('DELAY_DAYS', 'max'),
    INST_UNDERPAYMENT_FREQ=('UNDERPAID', 'mean'),
    INST_TOTAL_PAYMENTS=('AMT_PAYMENT', 'count'),
)
print(f"\n20% row sample: {_sample_mb:,.1f} MB "
      f"({(1 - _sample_mb / _native_mb) * 100:.1f}% smaller than native) -- more memory saved than downcasting, "
      f"but covers only {_agg_sample.shape[0] / _agg_full.shape[0] * 100:.1f}% of the full table's applicants")

_common = _agg_full.index.intersection(_agg_sample.index)
_cmp = _agg_full.loc[_common].join(_agg_sample.loc[_common], lsuffix='_full', rsuffix='_sample')
print("\nFidelity on applicants present in both (same people, fewer of their rows kept):")
_fidelity_rows = []
for _col in ['INST_DELAY_MEAN', 'INST_DELAY_MAX', 'INST_UNDERPAYMENT_FREQ', 'INST_TOTAL_PAYMENTS']:
    _f, _s = _cmp[f'{_col}_full'], _cmp[f'{_col}_sample']
    _valid = _f.notna() & _s.notna()
    _fidelity_rows.append({
        'feature': _col,
        'corr(full, sampled)': round(_f[_valid].corr(_s[_valid]), 3),
        'full_mean': round(_f[_valid].mean(), 2),
        'sampled_mean': round(_s[_valid].mean(), 2),
    })
print(pd.DataFrame(_fidelity_rows).to_string(index=False))

del _inst_full, _inst_sample, _agg_full, _agg_sample, _cmp
gc.collect()

Full table:   native 544.2 MB -> downcast 381.0 MB (30.0% smaller), 0% accuracy loss

20% row sample: 68.0 MB (87.5% smaller than native) -- more memory saved than downcasting, but covers only 94.2% of the full table's applicants

Fidelity on applicants present in both (same people, fewer of their rows kept):
               feature  corr(full, sampled)  full_mean  sampled_mean
       INST_DELAY_MEAN                0.699 -11.060000        -11.02
        INST_DELAY_MAX                0.523  16.780001          1.17
INST_UNDERPAYMENT_FREQ                0.825   0.090000          0.09
   INST_TOTAL_PAYMENTS                0.954  42.090000          8.51


18

In [11]:
## Pre-merge check: confirm each aggregated table has exactly one row per SK_ID_CURR
for _name, _agg in [('bureau_agg', bureau_agg), ('prev_agg', prev_agg),
                     ('pos_agg', pos_agg), ('inst_agg', inst_agg)]:
    n_dupes = _agg['SK_ID_CURR'].duplicated().sum()
    assert n_dupes == 0, f"{_name} has {n_dupes} duplicate SK_ID_CURR keys -- would inflate the merge"
    print(f"{_name}: {_agg.shape[0]:,} rows, {_agg['SK_ID_CURR'].nunique():,} unique keys -> OK, no duplicate matches")

bureau_agg: 305,811 rows, 305,811 unique keys -> OK, no duplicate matches
prev_agg: 338,857 rows, 338,857 unique keys -> OK, no duplicate matches
pos_agg: 337,252 rows, 337,252 unique keys -> OK, no duplicate matches
inst_agg: 339,587 rows, 339,587 unique keys -> OK, no duplicate matches


In [12]:
## join with SK_ID_CURR + merge onto the clean table
print(f"Base (Task 1) shape: {df_cleaned.shape}, {mb(df_cleaned):.1f} MB")

processed_features = (
    df_cleaned.merge(bureau_agg, on='SK_ID_CURR', how='left')
                .merge(prev_agg, on='SK_ID_CURR', how='left')
                .merge(pos_agg, on='SK_ID_CURR', how='left')
                .merge(inst_agg, on='SK_ID_CURR', how='left')
)

## rows with no match in behavioral table are NaN after the left join.

zero_fill_cols = [
    'BUREAU_ACTIVE_LINES', 'BUREAU_TOTAL_LINES', 'BUREAU_TOTAL_OVERDUE_DEBT', 'BUREAU_TOTAL_DEBT',
    'PREV_REFUSAL_COUNT', 'PREV_APPLICATION_COUNT',
    'POS_MONTH_COUNT', 'POS_COMPLETED_COUNT', 'POS_ACTIVE_COUNT',
    'INST_TOTAL_PAYMENTS',
]
processed_features[zero_fill_cols] = processed_features[zero_fill_cols].fillna(0)

processed_features['HAS_BUREAU_HISTORY'] = (processed_features['BUREAU_TOTAL_LINES'] > 0).astype('int8')

print(f"Final merged shape: {processed_features.shape}, {mb(processed_features):.1f} MB")

Base (Task 1) shape: (307511, 132), 364.9 MB
Final merged shape: (307511, 154), 414.4 MB


### Post-Merge Verification: Row-Count Integrity, Coverage & Missingness Semantics

Closing the loop on the join diagnostics above: confirm the merge neither dropped nor duplicated applicants, then report how many applicants actually matched each table (missing matches) now that the merge has happened.

In [13]:
## Row-count integrity: a left join on a unique key should never change the row count
assert processed_features.shape[0] == df_cleaned.shape[0], (
    f"Row count changed after merge: {df_cleaned.shape[0]:,} -> {processed_features.shape[0]:,} "
    "(duplicate or fanned-out keys in one of the aggregated tables)"
)
print(f"Row-count check OK: {processed_features.shape[0]:,} applicants before and after the merge.")

## Coverage: % of applicants with at least one record in each table, now measured directly on the merged matrix
_coverage = pd.DataFrame({
    'table': ['bureau', 'previous_application', 'POS_CASH_balance', 'installments_payments'],
    'has_history_count': [
        int((processed_features['BUREAU_TOTAL_LINES'] > 0).sum()),
        int((processed_features['PREV_APPLICATION_COUNT'] > 0).sum()),
        int((processed_features['POS_MONTH_COUNT'] > 0).sum()),
        int((processed_features['INST_TOTAL_PAYMENTS'] > 0).sum()),
    ],
})
_coverage['coverage_%'] = (_coverage['has_history_count'] / processed_features.shape[0] * 100).round(1)
print(_coverage.to_string(index=False))
print(
    f"\n{(processed_features['HAS_BUREAU_HISTORY'] == 0).sum():,} applicants "
    f"({(processed_features['HAS_BUREAU_HISTORY'] == 0).mean() * 100:.1f}%) are thin-file "
    "(zero bureau lines) -- this is Group B for the H2 subgroup test."
)

## (reuses zero_fill_cols from the merge cell above, so this stays in sync with what was actually filled)
_new_agg_cols = (set(bureau_agg.columns) | set(prev_agg.columns)
                 | set(pos_agg.columns) | set(inst_agg.columns)) - {'SK_ID_CURR'}
_informative_nan_cols = [c for c in processed_features.columns
                          if c in _new_agg_cols and c not in zero_fill_cols]
print(f"\nStructural-missing -> filled 0 ({len(zero_fill_cols)} cols): {zero_fill_cols}")
print(f"Informative-missing -> left NaN ({len(_informative_nan_cols)} cols): {_informative_nan_cols}")


Row-count check OK: 307,511 applicants before and after the merge.
                table  has_history_count  coverage_%
               bureau             263491        85.7
 previous_application             291057        94.6
     POS_CASH_balance             289444        94.1
installments_payments             291635        94.8

44,020 applicants (14.3%) are thin-file (zero bureau lines) -- this is Group B for the H2 subgroup test.

Structural-missing -> filled 0 (10 cols): ['BUREAU_ACTIVE_LINES', 'BUREAU_TOTAL_LINES', 'BUREAU_TOTAL_OVERDUE_DEBT', 'BUREAU_TOTAL_DEBT', 'PREV_REFUSAL_COUNT', 'PREV_APPLICATION_COUNT', 'POS_MONTH_COUNT', 'POS_COMPLETED_COUNT', 'POS_ACTIVE_COUNT', 'INST_TOTAL_PAYMENTS']
Informative-missing -> left NaN (11 cols): ['BUREAU_MAX_DAYS_OVERDUE', 'BUREAU_OLDEST_CREDIT_DAYS', 'PREV_APPROVAL_RATE', 'PREV_MEAN_DOWN_PAYMENT_PCT', 'POS_MOST_RECENT_MONTH', 'POS_MEAN_DPD', 'POS_MAX_DPD', 'POS_MEAN_DPD_DEF', 'INST_DELAY_MEAN', 'INST_DELAY_MAX', 'INST_UNDERPAYMENT_FREQ']

In [14]:
train_mask = processed_features['SPLIT'] == 'train'
train_df = processed_features.loc[train_mask].copy()

behavioral_cols = [
    'BUREAU_TOTAL_OVERDUE_DEBT', 'BUREAU_MAX_DAYS_OVERDUE', 'BUREAU_TOTAL_DEBT',
    'PREV_REFUSAL_COUNT', 'PREV_APPROVAL_RATE',
    'POS_MAX_DPD', 'POS_MEAN_DPD_DEF',
    'INST_DELAY_MEAN', 'INST_DELAY_MAX', 'INST_UNDERPAYMENT_FREQ',
]
corrs = (train_df[behavioral_cols + ['TARGET']]
         .corr(numeric_only=True)['TARGET'].drop('TARGET')
         .sort_values(key=abs, ascending=False))
print(corrs)

bins = [-np.inf, 0, 5, 15, np.inf]
labels = ['never late (<=0d)', '1-5d late', '6-15d late', '>15d late']
train_df['DELAY_BUCKET'] = pd.cut(train_df['INST_DELAY_MEAN'], bins=bins, labels=labels)
print(train_df.groupby('DELAY_BUCKET', observed=True)['TARGET'].agg(['mean', 'count']))

print(train_df.groupby('HAS_BUREAU_HISTORY')['TARGET'].agg(['mean', 'count']))

PREV_REFUSAL_COUNT           0.065928
PREV_APPROVAL_RATE          -0.065702
INST_UNDERPAYMENT_FREQ       0.061242
INST_DELAY_MEAN              0.022902
BUREAU_TOTAL_OVERDUE_DEBT    0.011347
POS_MEAN_DPD_DEF             0.008837
INST_DELAY_MAX               0.005095
POS_MAX_DPD                  0.005019
BUREAU_MAX_DAYS_OVERDUE      0.004002
BUREAU_TOTAL_DEBT            0.001708
Name: TARGET, dtype: float64
                       mean   count
DELAY_BUCKET                       
never late (<=0d)  0.080977  198735
1-5d late          0.110480    2815
6-15d late         0.141193    1459
>15d late          0.099595    1235
                        mean   count
HAS_BUREAU_HISTORY                  
0                   0.101378   30687
1                   0.077293  184570


**Reading the insight EDA (train split only, n=215,257):**
- **Prior refusals are a stronger signal than payment lateness.** `PREV_REFUSAL_COUNT` and
  `PREV_APPROVAL_RATE` have the largest correlations with `TARGET` (|r| ≈ 0.066) among all ten behavioral
  features tested -- a borrower's track record of getting *rejected* for credit predicts default slightly
  better than how late they later pay installments.
- **Default risk peaks in the middle of the lateness scale, not at the extreme.** Applicants averaging
  6-15 days late default at 14.1% -- nearly double the "never late" baseline of 8.1% -- but the *most*
  delinquent bucket (>15 days late) drops back down to 10.0%. This is a real, non-monotonic pattern
  (chronic-but-moderate delay carries more risk than one-off severe delay), not noise.
- **Thin-file applicants default 31% more often than bureau-backed ones in training data** (10.1% vs. 7.7%).
  This is the direct empirical motivation for Hypothesis 2 -- that alternative behavioral data should matter
  most exactly where bureau history is absent.

In [15]:
from sklearn.feature_selection import VarianceThreshold

numeric_cols = [c for c in processed_features.select_dtypes(include=[np.number]).columns
                if c not in ('SK_ID_CURR', 'TARGET')]
X_train_numeric = processed_features.loc[train_mask, numeric_cols].fillna(
    processed_features.loc[train_mask, numeric_cols].median())

vt = VarianceThreshold(threshold=0.0)
vt.fit(X_train_numeric)
zero_var_cols = [c for c, keep in zip(numeric_cols, vt.get_support()) if not keep]
print(f"{len(zero_var_cols)} zero-variance columns: {zero_var_cols}")

corr_matrix = X_train_numeric.corr(numeric_only=True).abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(a, b, upper.loc[b, a]) for a in upper.columns for b in upper.index
                    if pd.notna(upper.loc[b, a]) and upper.loc[b, a] > 0.90]
to_drop = sorted({b for a, b, v in high_corr_pairs})
print(f"{len(high_corr_pairs)} pairs with |r|>0.90 -> {len(to_drop)} candidate drop columns")

kept_cols = [c for c in numeric_cols if c not in zero_var_cols and c not in to_drop]
print(f"{len(numeric_cols)} numeric columns -> {len(kept_cols)} kept")

0 zero-variance columns: []
55 pairs with |r|>0.90 -> 39 candidate drop columns
134 numeric columns -> 95 kept


**Reading the feature-selection experiment (train split only):** zero zero-variance columns turned up,
which makes sense -- Task 1's cleaning already removed any constant columns. The real finding is the
correlation filter: 39 of 134 numeric columns (about 29%) are near-duplicates of another column at
|r| > 0.90. These aren't spurious -- they're almost entirely `*_AVG` / `*_MEDI` / `*_MODE` variants of the
same apartment/building attributes from `application_train.csv` (e.g. `FLOORSMAX_MEDI` ~ `FLOORSMAX_AVG`,
r=0.997), plus the `DAYS_EMPLOYED` / `DAYS_EMPLOYED_CLEAN` / `DAYS_EMPLOYED_ANOM` / `FLAG_EMP_PHONE` cluster
created during Task 1's pension-anomaly handling. A reduced 95-column matrix dropping these redundant
columns is a candidate for the modeling notebooks, without needing to re-run any model to justify it --
the redundancy is structural, not something cross-validation would need to discover.

In [16]:
## export to compressed parquet + document size reduction
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'processed_features.parquet'
processed_features.to_parquet(OUTPUT_PATH, engine='pyarrow', compression='snappy', index=False)

output_size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f"Exported {processed_features.shape[0]:,} rows x {processed_features.shape[1]} cols "
      f"-> {OUTPUT_PATH} ({output_size_mb:.1f} MB on disk)")

Exported 307,511 rows x 154 cols -> /Users/tungla/Desktop/UC Berkeley/DATASCI207_AppliedMachineLearning/DATASCI-207-FinalProject-Team1/DATASCI-207-FinalProject-Team1/data/processed/processed_features.parquet (37.3 MB on disk)
